# RetailPulse: AI-Powered Sales & Retail Analytics Platform
## Day 14 — Week 2 Checkpoint

**Module:** MLOps — Consolidation (Phase 3/4/5 wrap-up)

Per the execution plan, Day 14 consolidates Week 2: forecasting models, churn model, inventory optimization logic, hyperparameter tuning, drift detection, and the retraining pipeline should all be in place. This notebook:
1. Verifies every Week 2 artifact exists
2. Pulls a consolidated view across all three MLflow experiments
3. Produces a Week 2 status report

In [1]:
import os
import json
import pandas as pd
import mlflow

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
REPORTS_DIR = "../reports"
AIRFLOW_DIR = "../airflow_dags"

mlflow.set_tracking_uri("sqlite:///../mlflow.db")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Week 2 Artifact Inventory Check

In [2]:
expected_files = {
    "Forecasting (hybrid ensemble)": [
        f"{PROCESSED_DIR}/hybrid_forecast_30d.csv", f"{MODELS_DIR}/hybrid_metrics.json",
    ],
    "Churn model (base + tuned)": [
        f"{PROCESSED_DIR}/churn_scores.csv", f"{MODELS_DIR}/churn_xgboost.json",
        f"{MODELS_DIR}/churn_xgboost_tuned.json", f"{MODELS_DIR}/churn_metrics_tuned.json",
    ],
    "Inventory optimization": [
        f"{PROCESSED_DIR}/inventory_recommendations.csv",
    ],
    "Drift detection": [
        f"{PROCESSED_DIR}/drift_alert_status.json", f"{REPORTS_DIR}/day12_drift_report.html",
    ],
    "Retraining pipeline": [
        f"{AIRFLOW_DIR}/retailpulse_retraining_dag.py",
        f"{AIRFLOW_DIR}/retailpulse_pipeline/retrain.py",
        f"{PROCESSED_DIR}/retraining_run_summary.json",
    ],
}

all_ok = True
for category, files in expected_files.items():
    print(f"\n{category}:")
    for f in files:
        exists = os.path.exists(f)
        all_ok = all_ok and exists
        print(f"  [{'OK' if exists else 'MISSING'}] {f}")

print(f"\nWeek 2 artifact check: {'ALL PRESENT' if all_ok else 'INCOMPLETE'}")


Forecasting (hybrid ensemble):
  [OK] ../data/processed/hybrid_forecast_30d.csv
  [OK] ../models/hybrid_metrics.json

Churn model (base + tuned):
  [OK] ../data/processed/churn_scores.csv
  [OK] ../models/churn_xgboost.json
  [OK] ../models/churn_xgboost_tuned.json
  [OK] ../models/churn_metrics_tuned.json

Inventory optimization:
  [OK] ../data/processed/inventory_recommendations.csv

Drift detection:
  [OK] ../data/processed/drift_alert_status.json
  [OK] ../reports/day12_drift_report.html

Retraining pipeline:
  [OK] ../airflow_dags/retailpulse_retraining_dag.py
  [OK] ../airflow_dags/retailpulse_pipeline/retrain.py
  [OK] ../data/processed/retraining_run_summary.json

Week 2 artifact check: ALL PRESENT


## 2. Consolidated MLflow View — All Experiments

In [3]:
all_runs = []
for exp_name in ["RetailPulse-Forecasting", "RetailPulse-Churn", "RetailPulse-Retraining"]:
    try:
        runs = mlflow.search_runs(experiment_names=[exp_name])
        runs["experiment"] = exp_name
        all_runs.append(runs)
    except Exception as e:
        print(f"Could not load {exp_name}: {e}")

all_runs_df = pd.concat(all_runs, ignore_index=True)
cols = ["experiment", "tags.mlflow.runName", "metrics.mape", "metrics.auc_roc",
        "metrics.precision_at_top20"]
available_cols = [c for c in cols if c in all_runs_df.columns]
print(f"Total logged runs across all experiments: {len(all_runs_df)}")
all_runs_df[available_cols]

Total logged runs across all experiments: 6


,experiment,tags.mlflow.runName,metrics.mape,metrics.auc_roc,metrics.precision_at_top20
0,RetailPulse-Forecasting,day8_hybrid_ensemble,6.260,NaN,NaN
1,RetailPulse-Forecasting,day6_lstm_baseline,6.266,NaN,NaN
2,RetailPulse-Forecasting,day5_prophet_baseline,8.515,NaN,NaN
3,RetailPulse-Churn,day11_churn_xgboost_tuned,NaN,0.969382,1.0
4,RetailPulse-Churn,day9_churn_xgboost,NaN,0.964864,1.0
5,RetailPulse-Retraining,day13_automated_retrain_dryrun,NaN,NaN,NaN


## 3. Week 2 Status Report

In [4]:
with open(f"{MODELS_DIR}/hybrid_metrics.json") as f:
    hybrid = json.load(f)
with open(f"{MODELS_DIR}/churn_metrics_tuned.json") as f:
    churn = json.load(f)
inventory = pd.read_csv(f"{PROCESSED_DIR}/inventory_recommendations.csv")
with open(f"{PROCESSED_DIR}/drift_alert_status.json") as f:
    drift = json.load(f)
with open(f"{PROCESSED_DIR}/retraining_run_summary.json") as f:
    retrain_summary = json.load(f)

summary = f"""
WEEK 2 CHECKPOINT — RetailPulse: AI-Powered Sales & Retail Analytics Platform
{'='*70}

Forecasting (Day 5-8):
  - Hybrid ensemble MAPE : {hybrid['mape']:.2f}%  (target <=12%, met)
  - Blend weight         : Prophet={hybrid['best_prophet_weight']:.2f}, LSTM={1-hybrid['best_prophet_weight']:.2f}

Churn Prediction (Day 9, 11):
  - Tuned AUC-ROC        : {churn['auc_roc']:.4f}  (target >=0.88, met)
  - Precision@Top-20%    : {churn['precision_at_top20']:.4f}  (target >=0.75, met)
  - Optuna trials run    : 40 (CV AUC {churn['cv_auc']:.4f})

Inventory Optimization (Day 10):
  - SKUs analyzed        : {len(inventory)}
  - SKUs needing reorder : {(inventory['RecommendedOrderQty'] > 0).sum()}
  - ABC breakdown        : {inventory['ABC_Category'].value_counts().to_dict()}

MLOps (Day 12-13):
  - Drift detected       : {drift['retraining_recommended']}
  - Automated retrain run: models promoted = {retrain_summary['models_promoted']}
  - Total MLflow runs    : {len(all_runs_df)} across 3 experiments

Status: Week 2 objectives complete. Proceeding to Week 3
(Streamlit dashboard + analytics layer).
"""

print(summary)
with open(f"{REPORTS_DIR}/week2_checkpoint_summary.txt", "w") as f:
    f.write(summary)


WEEK 2 CHECKPOINT — RetailPulse: AI-Powered Sales & Retail Analytics Platform

Forecasting (Day 5-8):
  - Hybrid ensemble MAPE : 6.26%  (target <=12%, met)
  - Blend weight         : Prophet=0.05, LSTM=0.95

Churn Prediction (Day 9, 11):
  - Tuned AUC-ROC        : 0.9694  (target >=0.88, met)
  - Precision@Top-20%    : 1.0000  (target >=0.75, met)
  - Optuna trials run    : 40 (CV AUC 0.9573)

Inventory Optimization (Day 10):
  - SKUs analyzed        : 120
  - SKUs needing reorder : 42
  - ABC breakdown        : {'A': 67, 'B': 27, 'C': 26}

MLOps (Day 12-13):
  - Drift detected       : True
  - Automated retrain run: models promoted = ['prophet']
  - Total MLflow runs    : 6 across 3 experiments

Status: Week 2 objectives complete. Proceeding to Week 3
(Streamlit dashboard + analytics layer).



## 4. Day 14 Checkpoint Summary

**Outputs saved:**
- `reports/week2_checkpoint_summary.txt`

**Week 2 complete:** hybrid forecasting (Day 8), churn prediction + tuning (Day 9, 11), inventory optimization (Day 10), drift detection (Day 12), and an automated retraining pipeline with real promotion governance (Day 13) — all verified present and logged.

**Next module:** `15_dashboard_skeleton` — Week 3 begins, multi-page Streamlit dashboard.